In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

In [ ]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)

In [ ]:
SYSTEM_PROMPT = """You are an expert political-text annotator specializing in Nepali election manifestos and speeches.

TASK: Classify a given Nepali statement as either "C" for commitment or "NC" for non-commitment.

DEFINITIONS:

"commitment" - a promise made by a political leader/party before an election that is:
- Direct and clearly stated
- Specific and actionable (names a concrete policy, service, action, or mechanism)
- Has a clear, direct impact on a defined audience or issue
- NOT vague, aspirational, unfeasible, or overreaching

"non-commitment" - any statement that does NOT meet the above criteria. This includes TWO distinct types:

Type 1 (vague/aspirational): The statement uses future-tense promise-like language 
(e.g., "will be encouraged", "an environment will be created", "a policy will be taken") 
but does NOT specify any concrete action, mechanism, or implementation plan. 
It expresses a general intention or desired outcome without verifiable substance.
Example: "प्राङ्गारिक मलको प्रवर्धन र अर्गानिक खेतीको विकासलाई प्रोत्साहन गरिने छ।"
("Promotion of organic fertilizers and organic farming will be encouraged.")
Reasoning: The word "प्रोत्साहन" (encouragement) is a general intention with no 
concrete action or mechanism specified - not verifiable, so non-commitment.

Type 2 (out-of-context/unrelated): The statement is not a promise to the audience 
at all. This includes facts, historical narration, slogans, party/place/person names, 
descriptions, or criticism of opponents - regardless of whether it contains future-sounding words.
Example: "समृद्ध नेपाल, सुखी नेपाली" (a slogan, not a promise - no verb, no action)

INSTRUCTIONS:
- Read the statement carefully and apply the definitions above.
- Respond with EXACTLY ONE WORD: either "C" for commitment or "NC" for non-commitment.
- Do not provide any explanation, punctuation, or additional text."""


FEW_SHOT_EXAMPLES = [
    # ---- COMMITMENT (5) ----
    ("काठमाडौं महानगरपालिकामा आविष्कार केन्द्रका शाखाहरू सञ्चालनमा ल्याइने छन्।", "commitment"),
    ("बस्ती स्तर सम्म सडक संजाल नपुगेको ठाउँमा सडक पुऱ्याइने छ।", "commitment"),
    ("कार्यालयमा सेवामा आउने ज्येष्ठ नागरिक, अशक्त, अपाङ्गता भएका व्यक्तिहरुका लागि द्रुत सेवा लागू गरिने छ।", "commitment"),
    ("करदाताको विवरण तथा राजस्व सङ्कलन व्यवस्था सूचना प्रविधिमा आधारित राजस्व परिचालनको दीर्घकालिन योजना तर्जुमा, कार्यान्वयन र मुल्यांकन गरिनेछ।", "commitment"),
    ("युवाहरूलाई उनीहरुको व्यक्तिगत करियर मार्ग पत्ता लगाउन मद्‌दत गर्न करियर परामर्श र कोचिङ उपलब्धता गराइने छ।", "commitment"),

    # ---- NON-COMMITMENT: Type 2, out-of-context/factual (3) ----
    ("समृद्ध नेपाल, सुखी नेपाली", "non-commitment"),
    ("यस लिखु गाउँपालिका टोखा-छहरे-विदुर सडकको २६ कि.मी. दुरीमा अवस्थित रहेको छ।", "non-commitment"),
    ("समाजवादी प्रेस सङ्गठन, नेपाल नामक नयी सङ्गठन पनि निर्माण भएको छ।", "non-commitment"),

    # ---- NON-COMMITMENT: Type 1, vague/aspirational anomaly (2) ----
    ("सबै संस्थागत विद्यालय बीच एकरूपता कायम गरिने छ र स्वस्थ प्रतिस्पर्धाको वातावरण सृजना गरिने छ।", "non-commitment"),
    ("सबै किसिमका सहकारीको पूँजीलाई उत्पादन केन्द्रित बनाइने नीति लिइनेछ।", "non-commitment"),
]


In [ ]:
def build_messages(statement):

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for s, label in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": f"Statement: {s}"})
        messages.append({"role": "assistant", "content": label})

    messages.append(
        {"role": "user", "content": f"Statement: {statement}"}
    )

    return messages

In [ ]:
@torch.inference_mode()
def predict(statement):

    messages = build_messages(statement)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
    )

    pred = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    pred = pred.upper()

    if pred.startswith("C"):
        return "C"
    elif pred.startswith("NC"):
        return "NC"
    else:
        return "NC"


In [ ]:
df = pd.read_excel("Commitment-Mining/dataset/test_20p.xlsx")

texts = df["statements (ne)"].astype(str).tolist()
y_true = df["final_label"].str.strip().str.upper().tolist()

predictions = []

for i, text in enumerate(texts):

    pred = predict(text)
    predictions.append(pred)

    print(f"{i+1}/{len(texts)} -> {pred}")

df["prediction"] = predictions

df.to_excel("predictions.xlsx", index=False)

In [ ]:
np.save('y_pred-fs-llama-3.1.npy', predictions)

accuracy = accuracy_score( y_true,
    predictions)

precision = precision_score(
    y_true,
    predictions,
    average="macro",
)

recall = recall_score(
    y_true,
    predictions,
    average="macro",
)

f1 = f1_score(
    y_true,
    predictions,
    average="macro",
)

# Binary labels for AUROC
label_map = {"NC": 0, "C": 1}

y_true_bin = [label_map[x] for x in y_true]
y_pred_bin = [label_map[x] for x in predictions]

auroc = roc_auc_score(
    y_true_bin,
    y_pred_bin,
)


print(f"Accuracy : {accuracy:.4f}")
print(f"Macro Precision : {precision:.4f}")
print(f"Macro Recall    : {recall:.4f}")
print(f"Macro F1        : {f1:.4f}")
print(f"AUROC           : {auroc:.4f}")